# Lab 1B — Tool Calling, ReAct, and SQL Agents
**Day 1 Morning | ~45 minutes | Colab CPU | OpenAI API key required**

Part 1 treated the LLM as an inference engine: text in, text out. Part 2 changes the shape of the system.

The model can **ask your program to run a tool**. Your program does the work, sends the observation back, and the model writes the answer.

```
user question
    → model returns tool_calls (JSON), not a final answer
    → your Python runs the function
    → role=tool message with the result
    → model writes the grounded answer
```

We build that idea in four steps, each one a small win:

1. **One local tool** — memory sizing (the arithmetic from Lab 1A)
2. **One HTTP tool** — weather via Open-Meteo (no extra key)
3. **ReAct as a loop** — same pattern, old text format vs modern JSON
4. **SQL agent** — natural language → safe `SELECT` → explanation

> **The big idea:** the model never touches your database or APIs. You expose described tools. The model asks. Your code decides what actually runs.


---

## 0. Install and configure

Same secret as Lab 1A: `OPENAI_API_KEY`. Same `base_url` so this notebook stays compatible with a Lab 5 swap later.


In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} openai pandas httpx python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL   = "gpt-4o-mini"

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print(f"Ready — {DEFAULT_MODEL} at {OPENAI_BASE_URL}")

---

## Part A — From text generation to tool use

A normal completion asks the model to answer directly. Tool calling gives it another option: return a structured request such as

```json
{"name": "estimate_model_memory", "arguments": {"params_b": 7, "precision": "int4"}}
```

That JSON is **not** executed by the model. Your Python executes it.

| Who | Job |
|-----|-----|
| Model | Decides *which* tool and *which* arguments |
| Your app | Validates and runs the function |
| Model again | Reads the observation and writes the final answer |

If the schema is vague, the model uses the tool badly. The schema is part of your product.

### A tiny tool (no LLM yet)

This is the Lab 1A memory arithmetic, wrapped as a function the model will be allowed to call. Run it by itself first so you trust the numbers.


In [ ]:
import json

def estimate_model_memory(params_b: float, precision: str) -> str:
    bytes_per_param = {
        "fp32": 4.0,
        "fp16": 2.0,
        "bf16": 2.0,
        "int8": 1.0,
        "int4": 0.5,
        "nf4": 0.5,
    }
    key = precision.lower()
    if key not in bytes_per_param:
        return json.dumps({"error": f"Unsupported precision: {precision}"})
    memory_gb = params_b * bytes_per_param[key]
    return json.dumps(
        {
            "params_b": params_b,
            "precision": key,
            "estimated_weight_memory_gb": round(memory_gb, 2),
        }
    )

print(estimate_model_memory(7, "int4"))
print(estimate_model_memory(7, "fp8"))   # structured error, not a crash


**Checkpoint:** INT4 7B should print about **3.5 GB**. `fp8` should return an `error` JSON. Tools should fail as data, not as Python exceptions — otherwise the agent loop dies before the model can recover.

### Describe the tool to the model

The model cannot see your Python. It only sees this schema. `enum` lists legal values; `fp8` is included on purpose so we can demo a tool-side error later.


In [ ]:
memory_tool = {
    "type": "function",
    "function": {
        "name": "estimate_model_memory",
        "description": (
            "Estimate weight memory in GB for an LLM from parameter count and precision. "
            "Use this for deployment sizing questions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "params_b": {
                    "type": "number",
                    "description": "Model size in billions of parameters, such as 7 or 13.",
                },
                "precision": {
                    "type": "string",
                    "enum": ["fp32", "fp16", "bf16", "fp8", "int8", "int4", "nf4"],
                    "description": "Numeric precision used to store weights.",
                },
            },
            "required": ["params_b", "precision"],
        },
    },
}
print("Tool schema ready:", memory_tool["function"]["name"])


### The two-round loop, one cell at a time

`finish_reason` changes meaning:

- without tools → usually `stop` and a text answer (the model is **guessing**)
- with tools → may be `tool_calls` and **no** final answer yet (the model is **asking you to act**)


In [ ]:
question = "How much memory does a 7B INT4 model need?"

plain = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": question}],
)
print("WITHOUT tools")
print("  finish_reason:", plain.choices[0].finish_reason)
print("  content      :", plain.choices[0].message.content)


We now **force** a tool call with `tool_choice` so the demo is reliable. Later cells use `tool_choice='auto'` and let the model decide.


In [ ]:
tool_response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": question}],
    tools=[memory_tool],
    tool_choice={"type": "function", "function": {"name": "estimate_model_memory"}},
)

choice = tool_response.choices[0]
print("WITH tools (forced)")
print("  finish_reason:", choice.finish_reason)
print("  content      :", choice.message.content)
print("  tool_calls   :", choice.message.tool_calls)


**Checkpoint:** `finish_reason` should be `tool_calls`. `content` is often `None`. The arguments should look like `params_b=7`, `precision=int4`.

If the assistant message has **several** `tool_calls` (a comparison question), you must append one `role='tool'` message per `tool_call_id` before asking the model to continue.


In [ ]:
assistant_message = tool_response.choices[0].message
round2_messages = [
    {"role": "user", "content": question},
    assistant_message,
]

for tool_call in assistant_message.tool_calls:
    args = json.loads(tool_call.function.arguments)
    print("Model wants:", tool_call.function.name, args)
    if tool_call.function.name != "estimate_model_memory":
        tool_result = json.dumps({"error": f"Unknown tool: {tool_call.function.name}"})
    else:
        tool_result = estimate_model_memory(**args)
    print("Observation:", tool_result)
    round2_messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result,
        }
    )

final = client.chat.completions.create(model=DEFAULT_MODEL, messages=round2_messages)
print()
print("finish_reason:", final.choices[0].finish_reason)
print("Final answer :", final.choices[0].message.content)


Those three roles — `user`, `assistant` (with `tool_calls`), `tool` — are the protocol. LangChain, LlamaIndex, and AutoGen all assemble this structure under the hood.

Now wrap the loop **once** as `run_agent(question, tools, functions)`:

- `tools` — the list of schemas the model is allowed to choose from
- `functions` — a dict from tool name to the Python function that actually runs it

Parts B and D reuse this one function. Only the tools change.

In [ ]:
def run_agent(question, tools, functions, system="You answer LLM deployment questions. Use tools when they help."):
    """One tool-calling round trip: ask → run whatever tools the model requested → ask again."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]

    first = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages, tools=tools)
    reply = first.choices[0].message
    if not reply.tool_calls:                    # the model answered directly
        return reply.content

    messages.append(reply)
    for call in reply.tool_calls:
        args = json.loads(call.function.arguments)
        print(f"Tool requested: {call.function.name}({args})")
        result = functions[call.function.name](**args)
        print("Observation   :", result[:180])
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    second = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages)
    return second.choices[0].message.content

memory_tools = {"estimate_model_memory": estimate_model_memory}
print(run_agent("How much weight memory does a 7B model need in INT4 versus FP16?", [memory_tool], memory_tools))
print()
print(run_agent("How much memory does a 7B model need in fp8?", [memory_tool], memory_tools))

**Checkpoint:** the INT4 vs FP16 question should trigger **two** tool calls in one turn (parallel tool use). The fp8 question should surface the JSON error and still produce a sentence — the loop did not crash.

---

## Part B — A real HTTP tool: weather

Tools can call the network. Open-Meteo is free and needs **no API key**. The user says a city; the tool wants lat/long; the model fills coordinates from general knowledge; the tool returns live weather.

If the classroom network blocks the call, the function returns a mock JSON so you can keep going.


In [ ]:
import httpx

def get_current_weather(latitude: float, longitude: float) -> str:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": latitude, "longitude": longitude, "current_weather": True}
    try:
        response = httpx.get(url, params=params, timeout=10)
        response.raise_for_status()
        return response.text
    except Exception as exc:
        return json.dumps(
            {
                "error": f"Weather API unreachable ({exc.__class__.__name__})",
                "mock": True,
                "latitude": latitude,
                "longitude": longitude,
                "current_weather": {"temperature": 22.0, "windspeed": 8.0, "weathercode": 1},
            }
        )

weather_tool = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Get current weather for a latitude and longitude using Open-Meteo.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number", "description": "Latitude of the location."},
                "longitude": {"type": "number", "description": "Longitude of the location."},
            },
            "required": ["latitude", "longitude"],
        },
    },
}
print("Weather tool ready")


In [ ]:
print(run_agent("What is the current weather in Amman, Jordan?",
                tools=[weather_tool], functions={"get_current_weather": get_current_weather}))

**Checkpoint:** arguments should be Amman's coordinates (about 31.95 N, 35.93 E). The final answer should mention temperature. Same two-round loop as Part A — only the tool changed. The model chose the tool on its own this time (`tool_choice` defaulted to `auto`).

---

## Part C — ReAct is a loop, not a library

ReAct means **Reason + Act**:

```
Thought → Action → Observation → (repeat) → Answer
```

You already ran that loop in Parts A and B. Modern APIs return the Action as **JSON** (`tool_calls`). Older tutorials made the model **print** the action as text, then parsed it with regex.

Here is that old text format, using the **same** `estimate_model_memory` tool, so you can feel why JSON won. Bonus 03 walks a longer text-ReAct loop if you want more later.


In [ ]:
import re

text_react_system = '''You solve sizing questions using this format and nothing else:
Thought: <one sentence>
Action: estimate_model_memory: <params_b>, <precision>
PAUSE
Do not give a final Answer until you receive an Observation.'''

text_out = client.chat.completions.create(
    model=DEFAULT_MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": text_react_system},
        {"role": "user", "content": "How much memory does a 7B INT4 model need?"},
    ],
).choices[0].message.content

print("MODEL TEXT")
print(text_out)
print()

match = re.search(
    r"Action:\s*estimate_model_memory:\s*([0-9.]+),\s*(\w+)",
    text_out,
)
if not match:
    print("Could not parse an Action line. This is exactly why text-ReAct is fragile.")
else:
    params_b, precision = float(match.group(1)), match.group(2)
    observation = estimate_model_memory(params_b, precision)
    print("Parsed     :", params_b, precision)
    print("Observation:", observation)


| | Text-based ReAct (above) | Structured tools (Parts A–B) |
|---|---|---|
| Action format | `Action: name: args` | JSON `tool_calls` |
| Parsing | Regex — breaks if the model adds a period | API-level, typed arguments |
| Reliability | Classroom / old papers | What you ship |
| Where you will see it | Bonus 03, research blogs | OpenAI, Anthropic, Gemini, Groq |

The **idea** is identical. The **transport** is different. Prefer structured tools in production.

---

## Part D — SQL agent from scratch

Now the tool reaches a database, so the safety boundary matters.

The user asks about model benchmarks. The LLM does **not** know the table. It gets a `run_sql` tool and a schema description. Same loop: decide → your code validates and runs → observe rows → explain.

We use SQLite (built into Python) and the same Qwen sizes you will load in Labs 3–4, plus a few extras so queries are interesting.


In [ ]:
import os
import sqlite3
import pandas as pd

DB_PATH = "benchmarks.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute(
    "CREATE TABLE models ("
    "name TEXT, params_b REAL, vram_gb REAL, "
    "tokens_per_sec INTEGER, quality REAL)"
)
conn.executemany(
    "INSERT INTO models VALUES (?,?,?,?,?)",
    [
        ("Qwen2.5-0.5B", 0.5, 2.0, 120, 6.2),
        ("Qwen2.5-1.5B", 1.5, 4.0, 85, 7.1),
        ("Qwen2.5-7B", 7.0, 14.0, 40, 8.3),
        ("Llama-3.2-3B", 3.0, 6.0, 65, 7.4),
        ("Llama-3.1-8B", 8.0, 16.0, 35, 8.5),
        ("Mistral-7B", 7.0, 14.0, 42, 8.1),
    ],
)
conn.commit()
pd.read_sql_query("SELECT * FROM models", conn)


`benchmarks.db` is a file in the working directory (Colab: folder icon on the left). These numbers are **classroom fiction** for querying practice, not real published benchmarks.

### A safe SQL tool

Never hand an LLM a raw writeable connection. This classroom gatekeeper enforces:

- `SELECT` only
- one statement (no `;`)
- no comments
- only the `models` table
- max 20 rows, applied in Python
- errors returned as JSON


In [ ]:
import json
import re

def is_safe_select(query: str) -> tuple[bool, str]:
    normalized = " ".join(query.strip().split()).lower()
    if not normalized.startswith("select "):
        return False, "Only SELECT queries are allowed."
    blocked = [
        ";", "--", "/*", "*/",
        " pragma ", " attach ", " detach ",
        " drop ", " delete ", " update ", " insert ", " alter ", " create ",
    ]
    if any(token in f" {normalized} " for token in blocked):
        return False, "Query contains a blocked token or multiple-statement pattern."
    table_refs = re.findall(r"\b(?:from|join)\s+([a-zA-Z_][\w]*)", normalized)
    if not table_refs:
        return False, "Query must read from the models table."
    if any(table != "models" for table in table_refs):
        return False, "Query may only read from the models table."
    return True, "ok"


def run_sql(query: str) -> str:
    ok, reason = is_safe_select(query)
    if not ok:
        return json.dumps({"error": reason, "query": query})
    try:
        with sqlite3.connect(DB_PATH) as con:
            cur = con.execute(query)
            rows = cur.fetchmany(20)
            cols = [d[0] for d in cur.description]
            return json.dumps([dict(zip(cols, row)) for row in rows])
    except Exception as e:
        return json.dumps({"error": str(e), "query": query})

print("Allowed :", run_sql("SELECT name, vram_gb FROM models WHERE vram_gb <= 8 ORDER BY vram_gb"))
print("Blocked :", run_sql("SELECT name FROM models_v2"))


**Checkpoint:** the first print is real rows. The second is an error — `models_v2` is not allowlisted. The guardrail is doing its job.

The tool description must include **column names**. Without them the model guesses.


In [ ]:
sql_tool = {
    "type": "function",
    "function": {
        "name": "run_sql",
        "description": (
            "Run a read-only SQLite SELECT against the models table. "
            "Columns: name TEXT, params_b REAL, vram_gb REAL, "
            "tokens_per_sec INTEGER, quality REAL (0-10). "
            "SELECT only. No semicolons."
        ),
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "A single read-only SELECT over models."}},
            "required": ["query"],
        },
    },
}

def sql_agent(question):
    return run_agent(question, tools=[sql_tool], functions={"run_sql": run_sql},
                     system="You answer questions about LLM benchmark data. "
                            "Use run_sql when the answer needs table data. Cite the SQL evidence.")

print("sql_agent ready")

First, the same question **without** the tool — the model will sound confident and be guessing. Then with the agent, which has to look at *your* table.


In [ ]:
q = "Which models fit in 8 GB of VRAM?"

baseline = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": q}],
    temperature=0,
)
print("WITHOUT tool (guess from training data)")
print(baseline.choices[0].message.content)
print()
print("WITH sql_agent (queries benchmarks.db)")
print(sql_agent(q))


In [ ]:
for q in [
    "What is the fastest model with quality above 8.0?",
    "What is the average quality of models under 5B parameters?",
]:
    print("Q:", q)
    print(sql_agent(q))
    print("-" * 60)


**Checkpoint:** the 8 GB answer should name models from **your** table (Qwen2.5-0.5B / 1.5B / Llama-3.2-3B), not a random internet list.

---

## Exercise — add a second tool

Add `get_efficiency_score(name)` that looks up `quality` and `tokens_per_sec` for one model and returns

`round(quality / tokens_per_sec * 100, 2)`

as JSON. Then ask: **Which model has the best quality-per-speed efficiency?**

You will need a Python function, a tool schema, and one more entry in the `functions` dict you pass to `run_agent`.

In [ ]:
# TODO: implement get_efficiency_score, write its tool schema, then call run_agent with BOTH tools.

def get_efficiency_score(name: str) -> str:
    raise NotImplementedError("Query benchmarks.db and return JSON with an efficiency field")


# After you implement it:
# print(get_efficiency_score("Qwen2.5-1.5B"))
# print(run_agent("Which model has the best quality-per-speed efficiency?",
#                 tools=[sql_tool, efficiency_tool],
#                 functions={"run_sql": run_sql, "get_efficiency_score": get_efficiency_score}))

---

## Part E — What you just built

- **Tool calling** — the model selected a function and supplied JSON arguments
- **HTTP tool** — live (or mocked) weather through your code
- **ReAct** — thought → action → observation; JSON transport beats regex
- **SQL agent** — natural language became a gated `SELECT`

The model did not magically grow a database connection. **You** connected it with controlled tools. That is the deployment pattern.

Lab 2 will attack prompts (including injection). Lab 5 will put an OpenAI-compatible server in front of this same client. Lab 11 will add input/output guards around a RAG app that partners red-team in Lab 7.

### Production guardrails (read, do not implement today)

- Read-only database user
- Allowlist of tables and columns
- Real SQL parser, not only string checks
- Row limits and timeouts
- Log every generated query
- Human review for sensitive domains

### Lab 1B complete

- [ ] Memory-sizing agent (including a structured tool error)
- [ ] Weather tool call
- [ ] Clear picture of text-ReAct vs structured tools
- [ ] SQL agent over `benchmarks.db`
- [ ] (Stretch) `get_efficiency_score`

### Stretch

1. Add `cost_per_1k_tokens` and ask for the cheapest high-quality model.
2. Add `describe_schema()` and let the model inspect before querying.
3. Reject `SELECT *` in the validator.
4. After class, compare this scratch agent with a LangChain SQL agent (optional).
5. Ask `memory_agent('Compare 7B memory in INT4, FP16, and BF16')` and count `tool_calls` — that is parallel tool use.

Next: [Lab 2 — Prompting](../02_Prompting/README.md).
